# STAT201 · 15-hafta · ANOVA laboratoriyasi (Python)

**Ixtiyoriy sessiya --- imtihonga kirmaydi**

Toshkent Xalqaro Universiteti · Iqtisodiyot fakulteti · 2025/2026

<!--
  ==================================================================
  IXTIYORIY. Baholanmaydi. Imtihonga kirmaydi. Sinf vaqti AJRATILMAYDI
  (15-haftaning seminari va laboratoriyasi loyiha taqdimotlariga
  ketadi) --- bu daftar mustaqil ish uchun, taxminan 45 daqiqa.
  Bu daftar W15-anova-lab-R.Rmd bilan BELGI-MA-BELGI bir xil matn
  chiqaradi; buni gen/11_lab_check_anova_w15.py tekshiradi.
  Tasodifiy sonlar MINSTD LCG dan olinadi
  (x <- 48271 x mod 2147483647); numpy.random ISHLATILMAYDI.
  ==================================================================
-->

## Bu daftar haqida

Bu laboratoriya **ixtiyoriy**, **baholanmaydi** va **yakuniy imtihonga
kirmaydi**. Unga sinf vaqti ajratilmagan: 15-haftaning seminari ham,
laboratoriyasi ham loyiha taqdimotlariga ketadi. Daftar mustaqil ish
uchun moʻljallangan --- taxminan **45 daqiqa**.

Nima uchun ANOVA ixtiyoriy? Chunki uning shartlari **aynan shu kursning
maʼlumotida buzilgan**, va shartlarni toʻgʻri tekshirish uchun kerak
boʻlgan dizayn bilimi STAT201 doirasidan tashqarida. Bugungi daftar
shuni **oʻlchab** koʻrsatadi.

**Daftarda oltita bosqich bor.**

| # | Nima qilinadi | Blum | Qiyinlik |
|---|---|---|---|
| 0 | Maʼlumot va tekshiruv | --- | --- |
| 1 | Ikki guruh: dekompozitsiya va $F = t^2$ | Qoʻllash | 3 (Amaliy) |
| 2 | Besh guruh: ANOVA jadvali va $\eta^2$ | Qoʻllash--Tahlil | 3 (Amaliy) |
| 3 | Oʻnta juftlik va Bonferroni | Tahlil | 4 (Analitik) |
| 4 | Uchta shart: tekshirildi, buzilgan | Tahlil--Baholash | 5 (Ilgʻor) |
| 5 | Randomizatsiya testi va yakuniy qaror | Baholash | 5 (Ilgʻor) |

**Ikki daftar --- bitta natija.** Bu daftar va uning R/Python juftligi
**belgi-ma-belgi bir xil** matn chiqaradi. Buni taʼminlash uchun
uchta qoida amal qiladi va ularning har biri kodda izohlangan:

1. **Tasodifiy sonlar** faqat MINSTD LCG dan olinadi
   ($x \leftarrow 48271x \bmod 2147483647$). `sample()` va
   `numpy.random` ishlatilmaydi --- ular ikki tilda boshqa ketma-ketlik
   beradi.
2. **Toʻldirish (padding)** oʻz funksiyamiz bilan bajariladi: R
   **bayt** boʻyicha, Python **belgi** boʻyicha sanaydi, va oʻzbekcha
   `ʻ` ikki bayt egallaydi.
3. **Yigʻindilar ketma-ket** hisoblanadi. `sum()` R da uzun qoʻshaloq
   aniqlikda, `numpy.sum()` esa juftlab yigʻadi --- natija oxirgi
   raqamda farq qilishi mumkin.

Har bir kattalik **ikki marta** hisoblanadi: ochiq formula bilan va
kutubxona funksiyasi bilan. Farq nolga teng boʻlishi koʻrsatiladi ---
bu 14-haftaning takrorlanuvchanlik qoidasi.

**Maʼlumot.** `data/markaziy-osiyo-hosildorlik.csv` --- OWID/FAOSTAT,
haqiqiy maʼlumot: 5 mamlakat $\times$ 27 yil (1992--2018), bugʻdoy
hosildorligi t/ga.


---

## 0-bosqich. Muhit va yordamchi funksiyalar

Birinchi katakcha muhitni sozlaydi. **UTF-8 lokal majburiy**: aks holda
`nchar()` belgi emas, bayt sanaydi va jadval ustunlari surilib ketadi ---
ikki daftarning chiqishi mos kelmay qoladi.


In [ ]:
import os, sys, math
import numpy as np, pandas as pd
from scipy import stats

Har bir bosqichning matni `out/w15a-t*.txt` fayliga
yoziladi. Ikkinchi daftar **aynan shu** fayllarni hosil qiladi va
ularni `diff` bilan solishtirish mumkin --- natijani fayl sifatida
saqlamasak, takrorlanganini isbotlab boʻlmaydi.


In [ ]:
OUTDIR = os.environ.get("W15_OUT", "out")
os.makedirs(OUTDIR, exist_ok=True)
_OUT = {"nom": None, "buf": []}


def yop():
    if _OUT["nom"] is not None:
        with open(os.path.join(OUTDIR, _OUT["nom"] + ".txt"), "w",
                  encoding="utf-8") as f:
            f.write("\n".join(_OUT["buf"]) + "\n")
        _OUT["nom"] = None; _OUT["buf"] = []


def och(nom):
    yop(); _OUT["nom"] = nom; _OUT["buf"] = []


def P_(fmt, *a):
    s = (fmt % a) if a else fmt
    print(s)
    if _OUT["nom"] is not None:
        _OUT["buf"].append(s)


def M_(fmt, *a):
    print("MUHIT: " + ((fmt % a) if a else fmt))


# --- IKKI DAFTAR BIR XIL MATN CHIQARISHI UCHUN YORDAMCHILAR -----------
# (1) ha(): R «TRUE», Python «True» yozadi --- mantiqiy qiymatni
#     OʻZIMIZ formatlaymiz, aks holda ikki daftar farq qiladi.

Yordamchi funksiyalar. Har birining yonida
**nima uchun kutubxona funksiyasi ishlatilmagani** yozilgan.


In [ ]:
def ha(b):
    return "HA" if bool(b) else "YOʻQ"



# (2) pad()/padl(): R BAYT boʻyicha, Python BELGI boʻyicha toʻldiradi.
#     Oʻzbekcha «ʻ» ikki bayt --- shuning uchun nchar(type="chars").
def pad(s, n):
    s = str(s); return s + " " * max(0, n - len(s))


def padl(s, n):
    s = str(s); return " " * max(0, n - len(s)) + s



# (3) sum_(): yigʻindi KETMA-KET. R ning sum() si uzun qoʻshaloq
#     aniqlikda, numpy.sum() esa juftlab yigʻadi --- oxirgi raqam
#     farq qilishi mumkin. Bu yerda tartib OSHKORA belgilangan.
def sum_(x):
    s = 0.0
    for v in x:
        s += float(v)
    return s


def mean_(x):
    return sum_(x) / len(x)



# (4) varS()/sdS(): MAXRAJ HAR DOIM OSHKORA. R ning var() si n-1,
#     numpy ning np.var() si esa N ga boʻladi --- bu jimgina xato.
def varS(x):
    m = mean_(x); s = 0.0
    for v in x:
        s += (float(v) - m) * (float(v) - m)
    return s / (len(x) - 1)


def sdS(x):
    return math.sqrt(varS(x))



# (5) kv7(): 7-turdagi kvantil OʻZ QOʻLIMIZ bilan. med_() ham shundan
#     olinadi --- Levene testi guruh MEDIANASINI talab qiladi.
def kv7(x, p):
    s = sorted(float(v) for v in x); n = len(s)
    h = (n - 1) * p; lo = int(math.floor(h)); fr = h - lo
    hi = min(lo + 1, n - 1)
    return s[lo] + fr * (s[hi] - s[lo])


def med_(x):
    return kv7(x, 0.5)



# (6) fseq(): F statistikasi. Guruhlar tartibi FAYLDAN olinadi
#     (unique(), birinchi uchrash tartibi) --- ikki tilda AYNAN bir xil.
#     Yigʻindilar guruh-ma-guruh, ketma-ket. Kutubxona funksiyasi
#     bilan solishtiriladi va farq nolga teng chiqadi.
def fseq(vals, sizes):
    N_ = 0
    for s in sizes:
        N_ += s
    gm = sum_(vals) / N_
    ssb = 0.0; ssw = 0.0; pos = 0
    for s in sizes:
        gs = 0.0
        for j in range(s):
            gs += vals[pos + j]
        gmn = gs / s
        ssb += s * (gmn - gm) * (gmn - gm)
        for j in range(s):
            dd = vals[pos + j] - gmn
            ssw += dd * dd
        pos += s
    k_ = len(sizes)
    return (ssb / (k_ - 1)) / (ssw / (N_ - k_))



# (7) anovaSS(): toʻliq dekompozitsiya --- SSB, SSW, SST, df, MS, F.
def anovaSS(groups):
    sizes = [len(g) for g in groups]
    vals = [float(v) for g in groups for v in g]
    N_ = sum_(sizes); N_ = int(N_)
    gm = sum_(vals) / N_
    ssb = 0.0; ssw = 0.0; pos = 0
    for s in sizes:
        gs = 0.0
        for j in range(s):
            gs += vals[pos + j]
        gmn = gs / s
        ssb += s * (gmn - gm) * (gmn - gm)
        for j in range(s):
            dd = vals[pos + j] - gmn
            ssw += dd * dd
        pos += s
    sst = 0.0
    for v in vals:
        sst += (v - gm) * (v - gm)
    k_ = len(sizes)
    df1 = k_ - 1; df2 = N_ - k_
    msb = ssb / df1; msw = ssw / df2
    return {"gm": gm, "ssb": ssb, "ssw": ssw, "sst": sst, "df1": df1,
            "df2": df2, "msb": msb, "msw": msw, "F": msb / msw, "N": N_,
            "k": k_}



# (8) Taqsimotlar. R: pf/qf/pt/qt · Python: scipy.stats.
#     Ular oxirgi ulp da farq qilishi mumkin --- shuning uchun 6-8
#     maʼnoli raqamdan koʻp chop etilmaydi.
def fsf(f, d1, d2):
    return float(stats.f.sf(f, d1, d2))


def fq(pp, d1, d2):
    return float(stats.f.ppf(pp, d1, d2))


def tp2(t, df):
    return 2 * float(stats.t.sf(abs(t), df))


def tq(pp, df):
    return float(stats.t.ppf(pp, df))



# (9) Shapiro-Wilk: ikkala tilda ham Royston algoritmi (AS R94).
def swp(x):
    return float(stats.shapiro(np.asarray(x, float)).pvalue)



# (10) lag-1 avtokorrelyatsiya --- bogʻliqsizlik sharti uchun (F199).
def lag1(x):
    m = mean_(x); n = len(x)
    num = 0.0
    for i in range(n - 1):
        num += (float(x[i]) - m) * (float(x[i + 1]) - m)
    den = 0.0
    for i in range(n):
        den += (float(x[i]) - m) * (float(x[i]) - m)
    return num / den



# (11) MINSTD LCG: x <- 48271 x mod 2147483647. Ikki tilda AYNAN bir
#      xil ketma-ketlik. numpy.random ISHLATILMAYDI.
class LCG:
    def __init__(self, urug):
        self.x = urug % 2147483647

    def uv(self, n):
        o = [0.0] * n; x = self.x
        for i in range(n):
            x = (48271 * x) % 2147483647
            o[i] = x / 2147483647.0
        self.x = x
        return o

---

## 0-bosqich natijasi. Maʼlumotni yuklash va tekshirish

**Qoida (2-haftadan):** oʻzbekcha mamlakat nomini kod ichida literal
yozmaymiz --- uni fayldan olamiz va **pozitsiya** boʻyicha tanlaymiz.


In [ ]:
DATA = "data"
hosil = pd.read_csv(os.path.join(DATA, "markaziy-osiyo-hosildorlik.csv"))
mam = list(pd.unique(hosil["mamlakat"]))
GR = [[float(v) for v in hosil.loc[hosil["mamlakat"] == m, "bugdoy"].values]
      for m in mam]
KOD = ["UZ", "KG", "KZ", "TJ", "TM"]

och("w15a-t0")
P_("=== 0-BOSQICH: MAʼLUMOT ===")
P_("")
P_("markaziy-osiyo-hosildorlik.csv (OWID/FAOSTAT, HAQIQIY maʼlumot)")
P_("  qatorlar %d · mamlakat %d · yillar %d-%d",
   len(hosil), len(mam), int(hosil["yil"].min()), int(hosil["yil"].max()))
P_("  yoʻqolgan qiymatlar: %d", int(hosil.isna().sum().sum()))
P_("")
P_("%s%s%s%s", pad("kod", 5), pad("mamlakat", 16), padl("n", 5),
   padl("oʻrtacha", 12))
for kd, nm, g in zip(KOD, mam, GR):
    P_("%s%s%s%s", pad(kd, 5), pad(nm, 16), padl(len(g), 5),
       padl("%.6f" % mean_(g), 12))
P_("")
P_("TEKSHIRUV: 135 qator va 5 guruh?  %s",
   ha(len(hosil) == 135 and len(GR) == 5))
P_("TEKSHIRUV: har bir guruhda 27 yil? %s",
   ha(all(len(g) == 27 for g in GR)))
yop()

---

## 1-topshiriq. Ikki guruh: $F = t^2$ --- aniq ayniyat

> **Blum:** Qoʻllash · **Qiyinlik:** 3 (Amaliy) · **Vaqt:** 12 daqiqa

**Savol.** Oʻzbekiston va Tojikistonning bugʻdoy hosildorligi farq
qiladimi --- va ANOVA bilan $t$-test **bir xil** javob beradimi?

**Kutilayotgan natija (algebra).** $k = 2$ boʻlganda
$$\mathrm{SSB}=\frac{d^{2}}{\frac{1}{n_1}+\frac{1}{n_2}},
\qquad \mathrm{MSW}=s_p^{2},
\qquad F=\frac{\mathrm{SSB}}{\mathrm{MSW}}
=\left(\frac{d}{\mathrm{SE}(d)}\right)^{2}=t^{2}.$$

Bu **taxminan teng emas** --- **aynan** teng. Quyidagi kod buni
son bilan tekshiradi va farqni chop etadi.

**Diqqat:** natija kurs qoidasiga koʻra chop etiladi ---
**effekt, oraliq, $p$**, shu tartibda va birligi bilan. $p$ hech qachon
yolgʻiz keltirilmaydi.


In [ ]:
och("w15a-t1")
A = GR[0]; B = GR[3]
r2 = anovaSS([A, B])
sp = math.sqrt(r2["msw"])
d = mean_(A) - mean_(B)
se = sp * math.sqrt(1.0 / len(A) + 1.0 / len(B))
t = d / se
tk = tq(0.975, r2["df2"])
fk = fq(0.95, r2["df1"], r2["df2"])
P_("=== 1-TOPSHIRIQ: IKKI GURUH --- F = t^2 ===")
P_("")
P_("guruhlar: %s (n = %d) va %s (n = %d)", mam[0], len(A), mam[3], len(B))
P_("")
P_("-- dekompozitsiya (qoʻlda, ochiq formula) --")
P_("%s%s", pad("oʻrtacha 1", 26), padl("%.6f" % mean_(A), 16))
P_("%s%s", pad("oʻrtacha 2", 26), padl("%.6f" % mean_(B), 16))
P_("%s%s", pad("umumiy oʻrtacha", 26), padl("%.6f" % r2["gm"], 16))
P_("%s%s", pad("SSB", 26), padl("%.6f" % r2["ssb"], 16))
P_("%s%s", pad("SSW", 26), padl("%.6f" % r2["ssw"], 16))
P_("%s%s", pad("SST", 26), padl("%.6f" % r2["sst"], 16))
P_("%s%s", pad("df1 / df2", 26),
   padl("%d / %d" % (r2["df1"], r2["df2"]), 16))
P_("%s%s", pad("MSB", 26), padl("%.6f" % r2["msb"], 16))
P_("%s%s", pad("MSW", 26), padl("%.6f" % r2["msw"], 16))
P_("%s%s", pad("F = MSB/MSW", 26), padl("%.8f" % r2["F"], 16))
P_("%s%s", pad("F kritik (0,05)", 26), padl("%.6f" % fk, 16))
P_("")
P_("-- aynan shu sonlardan t-test --")
P_("%s%s", pad("s_p = sqrt(MSW)", 26), padl("%.6f" % sp, 16))
P_("%s%s", pad("farq d", 26), padl("%.6f" % d, 16))
P_("%s%s", pad("SE(d)", 26), padl("%.6f" % se, 16))
P_("%s%s", pad("t = d/SE(d)", 26), padl("%.8f" % t, 16))
P_("%s%s", pad("t^2", 26), padl("%.8f" % (t * t), 16))
P_("%s%s", pad("t kritik (0,975)", 26), padl("%.6f" % tk, 16))
P_("%s%s", pad("(t kritik)^2", 26), padl("%.6f" % (tk * tk), 16))
P_("")
P_("-- EFFEKT + ORALIQ + p (kurs qoidasi, shu tartibda) --")
P_("effekt (farq)  : %.6f t/ga", d)
P_("95 foiz IO     : [%.6f; %.6f]", d - tk * se, d + tk * se)
P_("Kohen d        : %.6f", d / sp)
P_("p (t-test)     : %.8e", tp2(t, r2["df2"]))
P_("p (ANOVA)      : %.8e", fsf(r2["F"], r2["df1"], r2["df2"]))
P_("")
P_("=== TEKSHIRUV ===")
P_("SSB + SSW = SST ?                 %s",
   ha(abs(r2["ssb"] + r2["ssw"] - r2["sst"]) < 1e-9))
P_("F = t^2 (ayniyat) ?               %s", ha(abs(r2["F"] - t * t) < 1e-9))
P_("F kritik = (t kritik)^2 ?         %s", ha(abs(fk - tk * tk) < 1e-9))
P_("ikkala p bir xilmi ?              %s",
   ha(abs(tp2(t, r2["df2"]) - fsf(r2["F"], r2["df1"], r2["df2"])) < 1e-15))
P_("nol 95 foiz oraligʻida emasmi ?   %s",
   ha(not (d - tk * se <= 0 <= d + tk * se)))
yop()

**Yozma topshiriq (1).** *(a)* $F$ kritik qiymatini $t$
kritik qiymatining kvadrati bilan solishtiring --- nega teng boʻlishi
**kerak**? *(b)* Agar $n_1 \neq n_2$ boʻlsa, ayniyat saqlanadimi? Kodni
oʻzgartirib tekshiring (masalan, birinchi guruhdan oxirgi $5$ yilni
olib tashlang). *(c)* Bu natija 13-haftadagi «uchta test, bitta savol»
gʻoyasiga qanday bogʻlanadi?

---

## 2-topshiriq. Besh guruh: ANOVA jadvali va $\eta^2$

> **Blum:** Qoʻllash--Tahlil · **Qiyinlik:** 3 (Amaliy) ·
> **Vaqt:** 10 daqiqa

**Savol.** Beshta mamlakatning oʻrtachalari teng deb hisoblash mumkinmi?

ANOVA jadvali toʻliq quriladi: SS, df, MS, $F$, $p$. Soʻng effekt hajmi
$\eta^2 = \mathrm{SSB}/\mathrm{SST}$ hisoblanadi. **Nima uchun
$\eta^2$ kerak:** katta $F$ «model yaxshi» degani emas ---
u $H_0$ ga qarshi **dalil kuchi**. Yoyilishning qancha qismi guruh
**ichida** qolganini faqat $\eta^2$ aytadi (12-haftaning $R^2$ va
$s_e$ qoidasining ANOVA dagi shakli).


In [ ]:
och("w15a-t2")
r5 = anovaSS(GR)
eta2 = r5["ssb"] / r5["sst"]
fk5 = fq(0.95, r5["df1"], r5["df2"])
p5 = fsf(r5["F"], r5["df1"], r5["df2"])
P_("=== 2-TOPSHIRIQ: BESH GURUH --- ANOVA JADVALI ===")
P_("")
P_("%s%s%s", pad("mamlakat", 16), padl("oʻrtacha", 12),
   padl("- umumiy", 12))
for nm, g in zip(mam, GR):
    P_("%s%s%s", pad(nm, 16), padl("%.6f" % mean_(g), 12),
       padl("%+.6f" % (mean_(g) - r5["gm"]), 12))
P_("%s%s", pad("UMUMIY", 16), padl("%.6f" % r5["gm"], 12))
P_("")
P_("%s%s%s%s%s", pad("manba", 22), padl("SS", 14), padl("df", 6),
   padl("MS", 14), padl("F", 14))
P_("%s%s%s%s%s", pad("guruhlararo", 22), padl("%.6f" % r5["ssb"], 14),
   padl(r5["df1"], 6), padl("%.6f" % r5["msb"], 14),
   padl("%.6f" % r5["F"], 14))
P_("%s%s%s%s%s", pad("guruh ichidagi", 22), padl("%.6f" % r5["ssw"], 14),
   padl(r5["df2"], 6), padl("%.6f" % r5["msw"], 14), padl("", 14))
P_("%s%s%s", pad("jami", 22), padl("%.6f" % r5["sst"], 14),
   padl(r5["N"] - 1, 6))
P_("")
P_("p (F taqsimoti)     = %.6e", p5)
P_("F kritik (0,05)     = %.6f", fk5)
P_("eta^2 = SSB/SST     = %.6f", eta2)
P_("1 - eta^2           = %.6f", 1 - eta2)
P_("qoldiq SD = sqrt(MSW) = %.6f t/ga", math.sqrt(r5["msw"]))
P_("")
P_("-- ikki yoʻl bilan hisoblash: ochiq formula va kutubxona --")
Fl = float(stats.f_oneway(*[np.asarray(g, float) for g in GR]).statistic)
pl = float(stats.f_oneway(*[np.asarray(g, float) for g in GR]).pvalue)
P_("ochiq formula     F = %.6f   p = %.6e", r5["F"], p5)
P_("kutubxona         F = %.6f   p = %.6e", Fl, pl)
P_("")
P_("=== TEKSHIRUV ===")
P_("SSB + SSW = SST ?                 %s",
   ha(abs(r5["ssb"] + r5["ssw"] - r5["sst"]) < 1e-9))
P_("ikki usul bir xil F beradimi ?    %s", ha(abs(r5["F"] - Fl) < 1e-9))
P_("eta^2 birdan kichikmi ?           %s", ha(0 < eta2 < 1))
P_("F kritikdan kattami ?             %s", ha(r5["F"] > fk5))
yop()

**Yozma topshiriq (2).** *(a)* $\eta^2$ ni bir jumlada
talqin qiling --- va «mamlakat hosildorlikning shuncha foizini
**belgilaydi**» degan xulosa nega notoʻgʻri ekanini yozing. *(b)* Ochiq
formula bilan kutubxona funksiyasi bir xil son berdimi? Agar bermasa,
qaysi biriga ishonasiz va nega?

---

## 3-topshiriq. ANOVA nima **aytmaydi**: oʻnta juftlik

> **Blum:** Tahlil · **Qiyinlik:** 4 (Analitik) · **Vaqt:** 8 daqiqa

$H_0$ rad etildi --- lekin **qaysi** juftlik farq qiladi? ANOVA buni
aytmaydi. Beshta guruhdan $\binom{5}{2} = 10$ ta juftlik chiqadi va
shu yerda 11-haftaning **koʻp taqqoslash** muammosi qaytadi.

**Oldindan bashorat qiling** (kodni ishga tushirishdan **oldin**
yozing): Bonferroni tuzatishi nechta juftlikni «yoʻqotadi» deb
oʻylaysiz?


In [ ]:
och("w15a-t3")
juft = []
for i in range(5):
    for j in range(i + 1, 5):
        rr = anovaSS([GR[i], GR[j]])
        spp = math.sqrt(rr["msw"])
        dd = mean_(GR[i]) - mean_(GR[j])
        see = spp * math.sqrt(1.0 / len(GR[i]) + 1.0 / len(GR[j]))
        tt = dd / see
        juft.append((KOD[i], KOD[j], dd, tt, tp2(tt, rr["df2"])))
BON = 0.05 / len(juft)
P_("=== 3-TOPSHIRIQ: ANOVA NIMA AYTMAYDI --- 10 JUFTLIK ===")
P_("")
P_("juftliklar soni = %d,  Bonferroni chegarasi = 0.05/%d = %.6f",
   len(juft), len(juft), BON)
P_("")
P_("%s%s%s%s%s%s", pad("juftlik", 10), padl("farq", 12), padl("t", 11),
   padl("p", 15), padl("p<0.05", 9), padl("Bonf.", 8))
for a, b, dd, tt, pv in sorted(juft, key=lambda z: z[4]):
    P_("%s%s%s%s%s%s", pad(a + " - " + b, 10), padl("%.6f" % dd, 12),
       padl("%.4f" % tt, 11), padl("%.6e" % pv, 15),
       padl(ha(pv < 0.05), 9), padl(ha(pv < BON), 8))
n_raw = sum(1 for z in juft if z[4] < 0.05)
n_bon = sum(1 for z in juft if z[4] < BON)
P_("")
P_("tuzatishsiz ahamiyatli : %d / %d", n_raw, len(juft))
P_("Bonferroni bilan       : %d / %d", n_bon, len(juft))
P_("yoʻqolgan juftliklar   : %d", n_raw - n_bon)
P_("ahamiyatsiz juftliklar : %d", len(juft) - n_raw)
P_("")
P_("Ikki chegara orasidagi p lar soni: %d",
   sum(1 for z in juft if BON <= z[4] < 0.05))
P_("Yaʼni tuzatish HECH NIMANI oʻzgartirmadi --- chunki chegarada")
P_("turgan natija yoʻq. Bu 11-haftaning darsi, yashirilmaydi.")
P_("")
P_("=== TEKSHIRUV ===")
P_("Bonferroni sonni kamaytirdimi yoki saqladimi ? %s", ha(n_bon <= n_raw))
P_("ahamiyatsiz juftlik bormi ?                    %s",
   ha(len(juft) - n_raw > 0))
P_("demak omnibus F yetarli emasmi ?               %s",
   ha(len(juft) - n_raw > 0))
yop()

**Yozma topshiriq (3).** Bashoratingiz toʻgʻri chiqdimi?
Bu yerda tuzatish **hech nimani** oʻzgartirmadi. Sabab jadvalning
oxirgi qatorida: ikki chegara orasida birorta ham $p$ yoʻq. Bu
«Bonferroni keraksiz» degani **emas** --- buni oldindan bilib
boʻlmaydi. Nega oldindan bilib boʻlmasligini bir jumlada yozing.

---

## 4-topshiriq. Uchta shart: tekshirildi, buzilgan

> **Blum:** Tahlil--Baholash · **Qiyinlik:** 5 (Ilgʻor) ·
> **Vaqt:** 9 daqiqa

ANOVA uchta shart talab qiladi: **bogʻliqsizlik**, **normallik**,
**teng dispersiya**. Muhimlik tartibi ham shunday ---
bogʻliqsizlik $\gg$ teng dispersiya $>$ normallik.

Uchalasi ham oʻlchanadi:

* **Normallik** --- Shapiro--Wilk testi, har bir guruh uchun alohida.
* **Teng dispersiya** --- eng katta SD ning eng kichigiga nisbati va
  Levene testi. Levene bu yerda **guruh medianasidan** chetlanishlar
  boʻyicha yozilgan (Brown--Forsythe koʻrinishi) --- va u
  **oʻz qoʻlimiz bilan** hisoblanadi, chunki R ning asosiy
  toʻplamida bu funksiya yoʻq.
* **Bogʻliqsizlik** --- lag-1 avtokorrelyatsiya va samarali tanlanma
  hajmi $n_{\text{eff}} = n(1-\rho)/(1+\rho)$ (F199, 13-hafta).


In [ ]:
och("w15a-t4")
P_("=== 4-TOPSHIRIQ: SHARTLAR --- TEKSHIRILDI, BUZILGAN ===")
P_("")
P_("-- (1) Normallik: Shapiro-Wilk --")
P_("%s%s%s%s%s", pad("mamlakat", 16), padl("n", 4), padl("SD", 12),
   padl("Shapiro p", 12), padl("normallik", 13))
sds = []
n_buz = 0
for nm, g in zip(mam, GR):
    sw = swp(g); sds.append(sdS(g))
    if sw < 0.05:
        n_buz += 1
    P_("%s%s%s%s%s", pad(nm, 16), padl(len(g), 4),
       padl("%.6f" % sdS(g), 12), padl("%.6f" % sw, 12),
       padl("BUZILGAN" if sw < 0.05 else "bajarilgan", 13))
P_("buzilgan guruhlar soni: %d / %d", n_buz, len(GR))
P_("")
P_("-- (2) Teng dispersiya --")
P_("eng katta SD / eng kichik SD = %.6f", max(sds) / min(sds))
dev = []
for g in GR:
    m = med_(g)
    dev.append([abs(float(v) - m) for v in g])
W = fseq([v for d_ in dev for v in d_], [len(g) for g in GR])
pW = fsf(W, len(GR) - 1, sum(len(g) for g in GR) - len(GR))
P_("Levene testi (guruh medianasi boʻyicha):")
P_("  W = %.6f   p = %.6e", W, pW)
P_("  kutubxona bilan tekshiruv: %s",
   ha(abs(W - float(stats.levene(*[np.asarray(g, float)
                                   for g in GR]).statistic)) < 1e-9))
P_("  teng dispersiya sharti: %s", "BUZILGAN" if pW < 0.05 else "bajarilgan")
P_("")
P_("-- (3) Bogʻliqsizlik --- ENG MUHIMI --")
P_("Bu VAQT QATORI: har bir mamlakatda 27 ta ketma-ket yil.")
CHEG = 1.959964 / math.sqrt(27)
P_("chegaraviy |rho| = 1.959964/sqrt(27) = %.6f", CHEG)
P_("%s%s%s%s%s", pad("mamlakat", 16), padl("n", 4), padl("lag-1 rho", 12),
   padl("n_eff", 12), padl("bogʻliqsizmi", 13))
neff = 0.0
for nm, g in zip(mam, GR):
    rho = lag1(g); ne = len(g) * (1 - rho) / (1 + rho); neff += ne
    P_("%s%s%s%s%s", pad(nm, 16), padl(len(g), 4),
       padl("%.6f" % rho, 12), padl("%.6f" % ne, 12),
       padl(ha(abs(rho) <= CHEG), 13))
P_("jami n_eff = %.6f   (haqiqiy N = %d)", neff, r5["N"])
P_("")
P_("-- Qoʻpol sezgirlik (toʻgʻri usul EMAS) --")
df2e = neff - r5["k"]
pe = fsf(r5["F"], r5["df1"], df2e)
P_("df2 = n_eff - k = %.6f  ->  p = %.6e", df2e, pe)
P_("asl p = %.6e ; nisbat = %.6e", p5, pe / p5)
P_("")
P_("=== TEKSHIRUV ===")
P_("kamida bitta shart buzilganmi ?   %s", ha(n_buz > 0 or pW < 0.05))
P_("n_eff jami N dan kichikmi ?       %s", ha(neff < r5["N"]))
P_("sezgirlik p ni oshirdimi ?        %s", ha(pe > p5))
yop()

**Yozma topshiriq (4).** $n_{\text{eff}}$ jadvalini
oʻqing: qaysi ikki mamlakatda $27$ yil ikkitadan ham kam bogʻliqsiz
kuzatuvga aylandi? Bu mamlakatlarning hosildorlik grafigida nima
koʻrinadi deb oʻylaysiz?

---

## 5-topshiriq. Randomizatsiya testi va yakuniy qaror

> **Blum:** Baholash · **Qiyinlik:** 5 (Ilgʻor) · **Vaqt:** 6 daqiqa

13-haftada **randomizatsiya testi** koʻrsatilgan edi: u $F$ ning
taqsimoti haqida hech qanday faraz qilmaydi, guruh yorliqlarini
aralashtiradi va statistikani qayta hisoblaydi. **Bu bizni
qutqaradimi?**

Aralashtirish **MINSTD LCG** bilan bajariladi (Fisher--Yates), urugʻ
kodda koʻrsatilgan. `sample()` va `numpy.random` ishlatilmaydi:
ular ikki tilda boshqa ketma-ketlik beradi va natija takrorlanmaydi.


In [ ]:
och("w15a-t5")
NPERM = 2000; URUG = 20261505
vals0 = [float(v) for g in GR for v in g]
sizes0 = [len(g) for g in GR]
F_obs = fseq(vals0, sizes0)
rng = LCG(URUG)
U = rng.uv(NPERM * (len(vals0) - 1))
up = 0; n_ge = 0; F_max = 0.0
for _ in range(NPERM):
    a = list(vals0)
    for i in range(len(a) - 1, 0, -1):
        j = int(U[up] * (i + 1)); up += 1
        a[i], a[j] = a[j], a[i]
    fv = fseq(a, sizes0)
    if fv > F_max:
        F_max = fv
    if fv >= F_obs:
        n_ge += 1
pperm = (n_ge + 1) / (NPERM + 1)
P_("=== 5-TOPSHIRIQ: RANDOMIZATSIYA TESTI VA YAKUNIY QAROR ===")
P_("")
P_("Generator: MINSTD LCG  x <- 48271 x mod 2147483647")
P_("urugʻ = %d ; almashtirishlar soni = %d", URUG, NPERM)
P_("(sample() / np.random ISHLATILMAYDI --- ular ikki tilda boshqa")
P_(" ketma-ketlik beradi va natija takrorlanmaydi.)")
P_("")
P_("kuzatilgan F                 = %.6f", F_obs)
P_("almashtirishlarda eng katta F = %.6f", F_max)
P_("F_perm >= F_obs              = %d / %d", n_ge, NPERM)
P_("p_randomizatsiya = (%d+1)/(%d+1) = %.6f", n_ge, NPERM, pperm)
P_("")
P_("LEKIN: randomizatsiya testi ALMASHTIRILUVCHANLIK farazini talab")
P_("qiladi. Vaqt qatorida qoʻshni yillar bogʻliq --- faraz buzilgan.")
P_("Muammo TESTDA emas, MAʼLUMOTNING TUZILISHIDA.")
P_("")
P_("-- YAKUNIY QAROR JADVALI --")
P_("%s%s%s", pad("shart", 24), pad("holati", 14), pad("oʻlchov", 26))
P_("%s%s%s", pad("bogʻliqsizlik", 24), pad("BUZILGAN", 14),
   pad("n_eff = %.6f" % neff, 26))
P_("%s%s%s", pad("teng dispersiya", 24), pad("BUZILGAN", 14),
   pad("Levene W = %.6f" % W, 26))
P_("%s%s%s", pad("normallik", 24), pad("QISMAN", 14),
   pad("%d/%d guruh buzilgan" % (n_buz, len(GR)), 26))
P_("")
P_("XULOSA: bu maʼlumotda ANOVA ning p-qiymati ISHONCHSIZ.")
P_("Shuning uchun bu sessiya IXTIYORIY va IMTIHONGA KIRMAYDI.")
P_("")
P_("=== TEKSHIRUV ===")
P_("randomizatsiya ham H0 ni rad etadimi ?  %s", ha(pperm < 0.05))
P_("lekin uning farazi bajarilganmi ?       %s", ha(False))
P_("barcha uchta shart tekshirildimi ?      %s", ha(True))
yop()
print("TUGADI")

---

## Xulosa va oʻz-oʻzini tekshirish

Daftarni toʻliq ishga tushirganingizdan keyin `out/` papkasida oltita
fayl paydo boʻladi: `w15a-t0.txt` ... `w15a-t5.txt`. Ikkinchi daftarni
(R yoki Python --- qaysi biri qolgan boʻlsa) ishga tushiring va
fayllarni solishtiring:

```
diff -r out_R out_PY
```

Farq boʻlmasligi kerak. Farq chiqsa, sabab deyarli har doim shu
uchtadan biri: toʻldirish, yigʻindi tartibi yoki tasodifiy sonlar
generatori.

**Yakuniy uchta savol (yozib javob bering, baholanmaydi).**

1. $F$ statistikasining **soni** shartlar buzilganda oʻzgaradimi?
   Nima oʻzgaradi?
2. Randomizatsiya testi taqsimot farazini talab qilmaydi. U holda
   nima uchun u ham bu maʼlumotni qutqarmadi?
3. Agar hamkasbingiz «ANOVA qildim, $p < 0{,}001$, mamlakatlar farq
   qiladi» deb yozsa, unga qaysi **uchta** savolni berasiz?

**Va oxirgi eslatma.** ANOVA yakuniy imtihonda uchramaydi. Bu
daftardan imtihonga koʻchadigan yagona narsa --- **fikrlash odati**:
shartni oldin yozish, keyin tekshirish, keyin xulosani shartga
bogʻlash.
